# 03 — Baseline Forecasts

The bar the ML models must clear. These are not straw men: seasonal naive exploits the strong weekly cycle, and Croston/SBA are the standard methods for the sparse series that dominate this panel.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
from src.data.loader import load_panel
cfg = load_config()
panel = load_panel(cfg)
print(f'panel: {len(panel):,} rows, {panel.series_id.nunique()} series, {panel.date.nunique()} days')

panel: 1,003,600 rows, 600 series, 1969 days


In [2]:
from src.forecasting.baselines import BASELINE_DESCRIPTIONS
import pandas as pd
pd.DataFrame(BASELINE_DESCRIPTIONS.items(), columns=['baseline','definition'])

,baseline,definition
0,naive,Repeat the last observed day (demand at the fo...
1,seasonal_naive,"Repeat the same weekday, stepped back in whole..."
2,moving_average_7,Mean of the last 7 days at the origin.
3,moving_average_28,Mean of the last 28 days at the origin.
4,croston,Croston's method: exponentially smoothed non-z...
5,sba,Syntetos-Boylan Approximation: Croston scaled ...
6,zero,Always forecast zero. Included because ~59% of...


## Rolling-origin evaluation of the baselines

In [3]:
from src.data.loader import rolling_origin_folds
for f in rolling_origin_folds(cfg, panel):
    print(f'{f.name}: origin {f.origin.date()} -> predict {f.start.date()}..{f.end.date()}')

fold_1: origin 2016-01-31 -> predict 2016-02-01..2016-02-28
fold_2: origin 2016-02-28 -> predict 2016-02-29..2016-03-27
fold_3: origin 2016-03-27 -> predict 2016-03-28..2016-04-24
fold_4: origin 2016-04-24 -> predict 2016-04-25..2016-05-22


In [4]:
from src.evaluation.validate import walk_forward, aggregate_scores, per_fold_table
from src.evaluation.metrics import skill_vs
res = walk_forward(cfg, panel, fit_ml=False)
skill_vs(aggregate_scores(res), 'seasonal_naive').round(4)

  fold_1   origin 2016-01-31 predict 2016-02-01..2016-02-28 | train rows 0 | best moving_average_28 WAPE 0.7635 | 0s


  fold_2   origin 2016-02-28 predict 2016-02-29..2016-03-27 | train rows 0 | best moving_average_7 WAPE 0.7776 | 0s


  fold_3   origin 2016-03-27 predict 2016-03-28..2016-04-24 | train rows 0 | best moving_average_28 WAPE 0.7463 | 0s


  fold_4   origin 2016-04-24 predict 2016-04-25..2016-05-22 | train rows 0 | best moving_average_28 WAPE 0.6977 | 0s


,model,folds,mae,rmse,wape,wape_std,mase,smape,bias,skill_vs_seasonal_naive_pct
0,moving_average_28,4,1.0455,2.2941,0.7503,0.0402,0.9492,132.0348,-0.0403,13.4363
1,moving_average_7,4,1.0536,2.2890,0.7558,0.0323,0.9582,115.4773,-0.0653,12.8049
2,sba,4,1.1001,2.3672,0.7901,0.0606,0.9796,142.9735,-0.0110,8.8489
3,croston,4,1.1150,2.3796,0.8008,0.0617,0.9941,142.2168,0.0619,7.6130
4,seasonal_naive,4,1.2083,2.7940,0.8668,0.0396,1.0670,80.1448,-0.0653,0.0000
5,zero,4,1.3958,3.8260,1.0000,0.0000,0.9420,87.8750,-1.3958,-15.3725
6,naive,4,1.4090,3.1630,1.0105,0.0817,1.1974,83.0502,0.3421,-16.5782


In [5]:
per_fold_table(res).round(4)

,fold_1,fold_2,fold_3,fold_4
naive,1.1160,1.0273,0.9259,0.9726
seasonal_naive,0.9051,0.8948,0.8442,0.8229
moving_average_7,0.7837,0.7776,0.7487,0.7131
moving_average_28,0.7635,0.7938,0.7463,0.6977
croston,0.8179,0.8703,0.7931,0.7218
sba,0.8094,0.8561,0.7838,0.7110
zero,1.0000,1.0000,1.0000,1.0000


## Croston is not a moving average

A naive implementation collapses to one: mean non-zero size x non-zero rate equals sum/n, the window mean. Real Croston smooths size and interval separately and updates only when a sale occurs. SBA then corrects Croston's known upward bias.

In [6]:
import numpy as np, pandas as pd
from src.features.build import croston_state
dates = pd.date_range('2020-01-01', periods=90, freq='D')
y = np.where(np.arange(90) % 5 == 0, 10.0, 0.0)
wide = pd.DataFrame([y], index=['S'], columns=dates)
print(croston_state(wide, alpha=0.1))
print('28-day moving average:', y[-28:].mean())

    croston       sba
S  2.307916  2.192521
28-day moving average: 1.7857142857142858


## Watch the `zero` baseline's MASE

Predicting zero everywhere attains the **best MASE** on this data, because averaging per-row scaled errors rewards predicting nothing on sparse series. WAPE correctly scores it 1.0. This is why WAPE is the headline metric here.